<a href="https://colab.research.google.com/github/ayushi777lodhi-stack/Vision-MultiModal-Learning/blob/main/timesformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torchvision timm einops

In [2]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from einops import rearrange
from tqdm import tqdm
import timm
from PIL import Image

In [3]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [4]:
!pip install kagglehub

In [5]:
class VideoFrame(Dataset):
  def __init__(self, root_dir, num_frames=8, image_size=224):
    self.root_dir=root_dir
    self.num_frames=num_frames
    self.classes=sorted(os.listdir(root_dir))
    self.class_to_idx={c: i for i,c in enumerate(self.classes)}
    self.samples=[]
    for cls in self.classes:
      cls_path=os.path.join(root_dir,cls)
      for video in os.listdir(cls_path):
        video_path=os.path.join(cls_path,video)
        if os.path.isdir(video_path):
            self.samples.append((video_path, self.class_to_idx[cls]))

    self.transform=transforms.Compose([transforms.Resize((image_size,image_size)),
                                       transforms.ToTensor()])

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    video_path, label=self.samples[idx]
    frames=sorted([f for f in os.listdir(video_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))])


    if len(frames)==0:
            raise RuntimeError(f"No images found in {video_path}")

    if len(frames)<self.num_frames:
      frames=frames*(self.num_frames//len(frames)+1)

    idxs=torch.linspace(0,len(frames)-1,self.num_frames).long()
    selected=[frames[i] for i in idxs]

    imgs=[]
    for f in selected:
      img=Image.open(os.path.join(video_path, f)).convert("RGB")
      imgs.append(self.transform(img))

    video=torch.stack(imgs)
    return video, label


In [6]:
class TransformerBlock(nn.Module):
  def __init__(self,dim,heads):
    super().__init__()

    self.temp_attn=nn.MultiheadAttention(dim,heads, batch_first=True)
    self.spat_attn=nn.MultiheadAttention(dim,heads, batch_first=True)

    self.norm1=nn.LayerNorm(dim)
    self.norm2=nn.LayerNorm(dim)
    self.norm3=nn.LayerNorm(dim)
    self.mlp=nn.Sequential(
        nn.Linear(dim,dim*4),
        nn.GELU(),
        nn.Linear(dim*4,dim))


  def forward(self,x):
    B,T,N,D=x.shape
    xt=rearrange(x, "b t n d -> (b n) t d")
    xt=self.temp_attn(xt,xt,xt)[0]
    xt=rearrange(xt,"(b n) t d -> b t n d", b=B, n=N)
    x=x+self.norm1(xt)
    xs=rearrange(x,"b t n d -> (b t) n d")
    xs=self.spat_attn(xs,xs,xs)[0]
    xs=rearrange(xs,"(b t) n d -> b t n d",b=B,t=T)
    x=x+self.norm2(xs)
    x=x+self.norm3(self.mlp(x))
    return x

In [ ]:
class TimeSformer(nn.Module):
  def __init__(self,num_classes,num_frames=8,patch_size=16,embed_dim=768,img_size=224,heads=12,depth=12):
    super().__init__()
    self.num_frames=num_frames
    self.patch_size=patch_size
    self.embed_dim=embed_dim
    self.num_patches=(img_size//patch_size)**2
    self.patch_embed=nn.Conv2d(3,embed_dim, kernel_size=patch_size, stride=patch_size)
    self.cls_token=nn.Parameter(torch.zeros(1,1,1,embed_dim))
    self.time_embed=nn.Parameter(torch.randn(1, self.num_frames+1,embed_dim))
    self.space_embed=nn.Parameter(torch.randn(1,self.num_patches,embed_dim))
    self.blocks=nn.ModuleList([
        TransformerBlock(embed_dim,heads)
        for _ in range(depth)])
    self.norm=nn.LayerNorm(embed_dim)
    self.head=nn.Linear(embed_dim, num_classes)

  def forward(self,x):
    B, T, C, H,W=x.shape
    x=x.view(B*T, C, H, W)
    x=self.patch_embed(x)
    x=x.flatten(2).transpose(1,2)
    x=x.view(B, T, -1, self.embed_dim)

    x=x+self.time_embed[:,1:T+1,None, :]+self.space_embed[:, None, :, :]

    cls=self.cls_token.expand(B, -1, self.num_patches,-1)
    cls=cls+self.time_embed[:,:1,None,:]

    x=torch.cat([cls,x], dim=1)
    for block in self.blocks:
      x=block(x)

    cls_out=self.norm(x[:,0,0])
    return self.head(cls_out)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# /content/drive

Mounted at /content/drive


In [ ]:
vit=timm.create_model("vit_base_patch16_224",pretrained=True)
vit.eval()

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [ ]:
def load_weights(timesformer, vit):
  timesformer.patch_embed.weight.data.copy_(vit.patch_embed.proj.weight.data)
  timesformer.patch_embed.bias.data.copy_(vit.patch_embed.proj.bias.data)

  for ts_block,vit_block in zip(timesformer.blocks, vit.blocks):
    ts_block.spat_attn.in_proj_weight.data.copy_(vit_block.attn.qkv.weight.data)
    ts_block.spat_attn.in_proj_bias.data.copy_(vit_block.attn.qkv.bias.data)
    ts_block.spat_attn.out_proj.weight.data.copy_(vit_block.attn.proj.weight.data)
    ts_block.spat_attn.out_proj.bias.data.copy_(vit_block.attn.proj.bias.data)
    ts_block.mlp[0].weight.data.copy_(vit_block.mlp.fc1.weight.data)
    ts_block.mlp[0].bias.data.copy_(vit_block.mlp.fc1.bias.data)
    ts_block.mlp[2].weight.data.copy_(vit_block.mlp.fc2.weight.data)
    ts_block.mlp[2].bias.data.copy_(vit_block.mlp.fc2.bias.data)
print("weights")

spatial weights loaded


In [ ]:
from torch.utils.data import random_split

dataset = VideoFrame(
    root_dir="/content/drive/MyDrive/datasettimes/output"
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
train_loader=DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2)

test_loader=DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2)

In [ ]:
dataset=train_dataset.dataset
model=TimeSformer(
    num_classes=len(dataset.classes),
    num_frames=8,
    embed_dim=768,
    depth=12,
    heads=12).to(device)

In [ ]:
load_weights(model,vit)
optimizer=torch.optim.AdamW(model.parameters(),lr=3e-5)
criterion=nn.CrossEntropyLoss()

In [ ]:
dataset = train_dataset.dataset

print(dataset.__dict__.keys())

dict_keys(['root_dir', 'num_frames', 'classes', 'class_to_idx', 'samples', 'transform'])


In [ ]:
print(type(train_dataset))
print(type(train_dataset.dataset))

<class 'torch.utils.data.dataset.Subset'>
<class '__main__.VideoFrame'>


In [ ]:
print(dataset.samples[0])

video_path, label = dataset.samples[0]
print(video_path)
print(os.listdir(video_path))

('/content/drive/MyDrive/datasettimes/output/bench press/video_002', 0)
/content/drive/MyDrive/datasettimes/output/bench press/video_002
['0012.jpg', '0017.jpg', '0005.jpg', '0015.jpg', '0019.jpg', '0006.jpg', '0030.jpg', '0013.jpg', '0011.jpg', '0002.jpg', '0016.jpg', '0010.jpg', '0004.jpg', '0021.jpg', '0014.jpg', '0018.jpg', '0025.jpg', '0029.jpg', '0032.jpg', '0031.jpg', '0023.jpg', '0024.jpg', '0020.jpg', '0001.jpg', '0028.jpg', '0009.jpg', '0027.jpg', '0007.jpg', '0008.jpg', '0022.jpg', '0026.jpg', '0003.jpg']


In [ ]:
epochs=10
def train(model,loader):
  model.train()
  total_loss=0
  correct=0
  total=0
  for epoch in range(1,epochs+1):
    for videos, labels in loader:
      videos=videos.to(device)
      labels=labels.to(device)

      logits=model(videos)
      loss=criterion(logits, labels)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      total_loss+=loss.item()
      preds=torch.argmax(logits,dim=1)
      correct+=(preds==labels).sum().item()
      total+=labels.size(0)

    avg_loss=total_loss/len(loader)
    accuracy=correct/total
    print(f"epoch: {epoch}/{epochs}")
    print(f"avg loss: {avg_loss} and accuracy:{accuracy}")

train(model,train_loader)

epoch: 1/10
avg loss: 0.8860718745838951 and accuracy:0.5
epoch: 2/10
avg loss: 1.5844724686765992 and accuracy:0.5
epoch: 3/10
avg loss: 2.2915748050688087 and accuracy:0.5405405405405406
epoch: 4/10
avg loss: 2.9473102703891896 and accuracy:0.5506756756756757
epoch: 5/10
avg loss: 3.665672206898799 and accuracy:0.5540540540540541
epoch: 6/10
avg loss: 4.302479351795203 and accuracy:0.5765765765765766
epoch: 7/10
avg loss: 4.876984806177584 and accuracy:0.5926640926640927
epoch: 8/10
avg loss: 5.461663451649852 and accuracy:0.6047297297297297
epoch: 9/10
avg loss: 5.9618714497097445 and accuracy:0.6216216216216216
epoch: 10/10
avg loss: 6.495820684333307 and accuracy:0.6378378378378379


In [ ]:
def test(model, loader):
    model.eval()
    total_loss=0
    correct=0
    total=0

    with torch.no_grad():

        for videos,labels in loader:
            videos=videos.to(device)
            labels=labels.to(device)
            outputs=model(videos)
            loss=criterion(outputs,labels)
            total_loss+=loss.item()
            preds=outputs.argmax(dim=1)
            correct+=(preds==labels).sum().item()
            total+=labels.size(0)

    avg_loss=total_loss/len(loader)
    accuracy=100*correct/total
    print(f"Test Loss : {avg_loss:.4f}")
    print(f"Test Accuracy : {accuracy:.2f}%")

    return avg_loss, accuracy

In [ ]:
test_loss,test_acc=test(model, test_loader)

Test Loss : 0.3875
Test Accuracy : 84.21%


In [ ]:
model.eval()
all_preds=[]
all_labels=[]

with torch.no_grad():

    for videos,labels in test_loader:
        videos=videos.to(device)
        labels=labels.to(device)
        outputs=model(videos)
        preds=outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Ground Truth:",all_labels)
print("Predictions:",all_preds)

Ground Truth: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
Predictions: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
